# Modelo de compra (Fase 1), criterio *equilibrado*, con la restricción (4) manejada explícitamente

Para cada polígono $p$ se maximiza la **cobertura mínima** entre especies, $\min_e\big(1-\mathbb{E}[F_{e,p}]/D_{e,p}\big)$, con $F_{e,p}=\max\{0,D_{e,p}-X_{e,p}-Z_{e,p}\}$, sujeto a

$$\sum_e c_e(q^I_{e,p}+q^R_{e,p})\le B A_p\ (3),\quad \sum_e q^I_{e,p}\le \mathbb{E}[L_p]\ (4),\quad q^I_{e,p}\le \max\{0,(1+\delta)D_{e,p}-\mathbb{E}[X_{e,p}]\}\ (5),\quad q^R_{e,p}\le q^I_{e,p}\ (6).$$

$X$: modelo B, Bin$(x,0.95)$. $Z$: secuencial con $S_e$ a 2 años. Números aleatorios comunes (un flujo de Bernoulli por planta).

**Cómo se maneja (4).** Sembrar y reponer cuestan lo mismo y aportan lo mismo mientras haya cepas muertas, pero solo sembrar usa un punto de la malla. Cuando los puntos escasean, conviene reponer más. El algoritmo asigna un **precio $\lambda$ por punto de malla** (pesos): el puntaje de una siembra es el beneficio dividido entre $c_e+\lambda$, y el de una reposición entre $c_e$. Se prueban varios $\lambda$ por polígono y se conserva el que logra mayor cobertura mínima; si con $\lambda=0$ ya se alcanza la cota del presupuesto, no se busca más.

**Cotas.** Cada planta comprada aporta a lo más $S_e$ plantas establecidas en esperanza, lo que da una cota de cobertura mínima que solo usa el presupuesto. Si además se exige $\sum q^I\le L_p$ y $q^R\le(1-S_e)q^I$ en esperanza, la cota se calcula con un LP por polígono (`cota_lp`). Sirven para saber cuánto se puede mejorar como máximo.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linprog

# ---------------- CONFIGURACIÓN ----------------
N_OPT, N_VAL, SEED = 10_000, 20_000, 2026
ERR_REG = 0.05            # modelo B
DELTA   = 0.05            # restricción (5)
PREF_R  = 0.99            # desempate: a igualdad se prefiere sembrar antes que reponer
LAMS    = (0.0, 60.0, 200.0)   # precios de punto de malla que se prueban ($ por siembra)
TOL     = 0.003           # si la cobertura mínima llega a la cota menos TOL, no se busca más
SUPERV  = 'sobrev_2023'

datos = pd.read_csv('datos_limpios.csv')
par   = pd.read_csv('parametros_especies.csv').set_index('especie')
esp   = list(par.index)
x     = datos.pivot(index='especie', columns='poligono', values='plantas').loc[esp]
A     = datos.drop_duplicates('poligono').set_index('poligono')['area_ha'].sort_index()
x     = x[A.index]
E_, P_ = x.shape
d = par['meta_plantas_ha'].values.astype(float); c = par['costo_c'].values.astype(float); S = par[SUPERV].values.astype(float)
D = np.outer(d, A.values)

def simular_X(n, rng):
    return rng.binomial(x.values, 1 - ERR_REG, size=(n, E_, P_))

X   = simular_X(N_OPT, np.random.default_rng(SEED))
EXm = X.mean(0)
print('X:', X.shape, '| meta total:', f'{D.sum():,.0f}', 'plantas | área total:', round(A.sum(), 1), 'ha')

## 1. Optimizador por polígono y cotas

In [ ]:
def optimizar_poligono(p, X, rng, B_ha, ptos_ha, lam=0.0, delta=DELTA):
    """Criterio equilibrado para el polígono p (0-based). lam: precio de un punto de malla ($ por siembra)."""
    n = X.shape[0]; idx = np.arange(n)
    EX    = X[:, :, p].mean(0)
    Np_p  = int(round(ptos_ha * A.values[p]))
    L     = np.maximum(Np_p - X[:, :, p].sum(1), 0)
    slots = int(np.floor(L.mean()))                                           # restricción (4)
    U     = np.maximum(0, np.floor((1 + delta) * D[:, p] - EX)).astype(int)   # restricción (5)
    budget = B_ha * A.values[p]                                               # restricción (3)

    B1 = [(rng.random((U[e], n)) < S[e]).astype(np.int8) for e in range(E_)]
    C2 = [np.vstack([np.zeros((1, n), np.int16),
                     np.cumsum(rng.random((U[e], n)) < S[e], axis=0, dtype=np.int16)]) for e in range(E_)]
    qI = np.zeros(E_, int); qR = np.zeros(E_, int)
    ZI = np.zeros((E_, n), np.int32); dead = np.zeros((E_, n), np.int32)
    Fm = np.maximum(0.0, D[:, p, None] - X[:, :, p].T).mean(1)

    def candidatos(e):
        out = [-np.inf, -np.inf]; new = [None, None]
        if qI[e] < U[e]:                                                       # +1 siembra
            zi = ZI[e] + B1[e][qI[e]]; dd = (qI[e] + 1) - zi
            zr = C2[e][np.minimum(qR[e], dd), idx]
            F2 = np.maximum(0.0, D[e, p] - X[:, e, p] - zi - zr)
            out[0] = (Fm[e] - F2.mean()) / (c[e] + lam); new[0] = (zi, dd, F2)
        if qR[e] < qI[e]:                                                      # +1 reposición (restricción 6)
            zr = C2[e][np.minimum(qR[e] + 1, dead[e]), idx]
            F2 = np.maximum(0.0, D[e, p] - X[:, e, p] - ZI[e] - zr)
            out[1] = PREF_R * (Fm[e] - F2.mean()) / c[e]; new[1] = (ZI[e], dead[e], F2)
        return out, new

    sc = np.full((E_, 2), -np.inf); nw = [[None, None] for _ in range(E_)]
    for e in range(E_): sc[e], nw[e] = candidatos(e)
    left, slots_left = budget, slots
    while True:
        ok = sc.max(1) > 1e-12
        if not ok.any(): break
        cob = np.where(ok, 1 - Fm / D[:, p], np.inf)                           # cobertura actual por especie
        e = int(np.argmin(cob)); t = int(np.argmax(sc[e]))
        if c[e] > left:                sc[e] = -np.inf; continue
        if t == 0 and slots_left <= 0: sc[e, 0] = -np.inf; continue
        zi, dd, F2 = nw[e][t]
        if t == 0: qI[e] += 1; slots_left -= 1
        else:      qR[e] += 1
        ZI[e], dead[e] = zi, dd; Fm[e] = F2.mean(); left -= c[e]
        sc[e], nw[e] = candidatos(e)
    return qI, qR, dict(gasto=budget - left, presupuesto=budget, slots=slots - slots_left, slots_disp=slots, EF=Fm.copy(), lam=lam)

def cota_presupuesto(p, B_ha):
    """Cobertura mínima máxima con solo el presupuesto (cada planta aporta a lo más S_e en esperanza)."""
    return (B_ha * A.values[p] + (c * EXm[:, p] / S).sum()) / (c * D[:, p] / S).sum()

def cota_lp(p, B_ha, ptos_ha):
    """Cota con presupuesto y puntos de malla: siembras >= Q_e/(2-S_e) (reposición <= (1-S_e)*siembras)."""
    L = ptos_ha * A.values[p] - EXm[:, p].sum()
    cv = np.zeros(E_ + 1); cv[-1] = -1
    Au = [np.r_[c, 0], np.r_[1 / (2 - S), 0]]; bu = [B_ha * A.values[p], L]
    for e in range(E_):
        r = np.zeros(E_ + 1); r[e] = -S[e]; r[-1] = D[e, p]; Au.append(r); bu.append(EXm[e, p])
    res = linprog(cv, A_ub=np.array(Au), b_ub=np.array(bu), bounds=[(0, None)] * E_ + [(0, 1)], method='highs')
    return -res.fun

def resolver_todos(X, B_ha, ptos_ha, refinar=True):
    QI = np.zeros((E_, P_), int); QR = np.zeros((E_, P_), int); infos = []
    for p in range(P_):
        mejor = None
        for lam in (LAMS if refinar else (0.0,)):
            qi, qr, info = optimizar_poligono(p, X, np.random.default_rng(1000 + p), B_ha, ptos_ha, lam)
            cob = 1 - info['EF'] / D[:, p]
            clave = (round(cob.min(), 4), -info['EF'].sum())
            if mejor is None or clave > mejor[0]: mejor = (clave, qi, qr, info)
            if cob.min() >= cota_lp(p, B_ha, ptos_ha) - TOL: break               # ya en la cota: no hay nada que mejorar
        QI[:, p], QR[:, p] = mejor[1], mejor[2]; infos.append(mejor[3])
    return QI, QR, infos

## 2. ¿Desde cuántos puntos de malla por ha limita la restricción (4)?
Cobertura mínima máxima (mediana de los 30 polígonos) según la cota solo con presupuesto y la cota LP que incluye los puntos de malla.

In [ ]:
filas = []
for B_ha in (12_500, 13_200):
    base = np.median([cota_presupuesto(p, B_ha) for p in range(P_)])
    for ptos in (540, 560, 580, 600, 620, 640, 658):
        lp = np.median([cota_lp(p, B_ha, ptos) for p in range(P_)])
        filas.append((B_ha, ptos, 100 * base, 100 * lp, 'limita' if lp < base - 0.002 else 'no limita'))
pd.DataFrame(filas, columns=['presupuesto_ha', 'puntos_malla_ha', 'cota_solo_presupuesto_%', 'cota_con_malla_%', '(4)']).round(2)

## 3. Validación fuera de muestra
20,000 escenarios nuevos de $X$ y $Z$ (simulación directa).

In [ ]:
def validar(QI, QR, n=N_VAL, seed=99):
    r = np.random.default_rng(seed); Xv = simular_X(n, r)
    ZI = r.binomial(QI, S[:, None], size=(n, E_, P_)); ZR = r.binomial(np.minimum(QR, QI - ZI), S[:, None])
    Z = ZI + ZR; return Xv, Z, np.maximum(0.0, D - Xv - Z)

def resumen(QI, QR):
    Xv, Z, F = validar(QI, QR)
    t = pd.DataFrame({'meta': D.sum(1), 'E_X': Xv.mean(0).sum(1), 'E_Z': Z.mean(0).sum(1), 'E_F': F.mean(0).sum(1),
                      'q_siembra': QI.sum(1), 'q_reposicion': QR.sum(1), 'gasto': (c[:, None] * (QI + QR)).sum(1)}, index=esp)
    t['cobertura_%'] = 100 * (1 - t.E_F / t.meta)
    return t

escenarios = [('A', 12_500, 658, False), ('B', 13_200, 658, False), ('C0', 13_200, 600, False), ('C', 13_200, 600, True)]
resultados, filas = {}, []
for nombre, B_ha, ptos, refinar in escenarios:
    t0 = time.time(); QI, QR, infos = resolver_todos(X, B_ha, ptos, refinar)
    t = resumen(QI, QR); resultados[nombre] = (B_ha, ptos, QI, QR, t)
    cota = np.average([cota_lp(p, B_ha, ptos) for p in range(P_)], weights=D.sum(0))
    filas.append((nombre, B_ha, ptos, 'con precio λ' if refinar else 'λ = 0 (anterior)', time.time() - t0,
                  t.E_F.sum(), 100 * t.E_F.sum() / t.meta.sum(), t['cobertura_%'].min(), 100 * cota,
                  t.gasto.sum(), B_ha * A.sum(), sum(i['slots'] for i in infos), sum(i['slots_disp'] for i in infos)))
    print(f'{nombre}: {time.time()-t0:.0f} s', flush=True)
comp = pd.DataFrame(filas, columns=['esc', 'presup_ha', 'puntos_ha', 'método', 'seg', 'faltante_esperado', 'faltante_%',
                                    'cobertura_min_%', 'cota_LP_%', 'gasto', 'presupuesto', 'puntos_usados', 'puntos_disp']).set_index('esc')
comp.round({'seg': 0, 'faltante_esperado': 0, 'faltante_%': 2, 'cobertura_min_%': 2, 'cota_LP_%': 2, 'gasto': 0, 'presupuesto': 0})

`A` y `B` usan 658 puntos/ha (placeholder); `C0` y `C` usan una malla más ajustada de 600 puntos/ha, sin y con precio de punto de malla. La cobertura mínima está en la validación fuera de muestra, y la cota LP es el máximo alcanzable en esperanza (promedio ponderado por meta de los polígonos).

In [ ]:
cob = pd.DataFrame({k: v[4]['cobertura_%'] for k, v in resultados.items()}).round(1)
display(cob)
ax = cob.plot.bar(figsize=(11, 4.5), width=0.85); ax.set_ylabel('Cobertura esperada de la meta (%)'); ax.set_xlabel('')
ax.set_title('Cobertura por especie (criterio equilibrado)'); plt.xticks(rotation=35, ha='right'); plt.legend(title='escenario', fontsize=8)
plt.tight_layout(); plt.show()

## 4. Plan de compra por polígono y especie

In [ ]:
filas = []
for nombre in ('A', 'B', 'C'):
    B_ha, ptos, QI, QR, _ = resultados[nombre]
    for j, pol in enumerate(A.index):
        for i, e in enumerate(esp):
            filas.append((nombre, B_ha, ptos, pol, A.values[j], e, QI[i, j], QR[i, j], QI[i, j] + QR[i, j], c[i] * (QI[i, j] + QR[i, j])))
plan = pd.DataFrame(filas, columns=['escenario', 'presupuesto_ha', 'puntos_malla_ha', 'poligono', 'area_ha', 'especie',
                                    'q_siembra', 'q_reposicion', 'q_total', 'costo'])
plan.to_csv('plan_compra_equilibrado.csv', index=False); print(plan.shape)
plan.query("escenario == 'B' and poligono == 1")